<a href="https://colab.research.google.com/github/jaiswalvik/MathematicalFoundationsOfGenerativeAI/blob/master/image_classification_butterfly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦋 Butterfly Image Classification
### Transfer learning with ResNet18, ResNet34, and VGG19

This notebook is organized as a guided workflow: **prepare data → configure transforms → build models → train/evaluate → generate test predictions**.

**Before you begin**
- Run the cells from top to bottom in Google Colab.
- Select **Runtime → Change runtime type → T4 GPU** for faster training.
- The test CSV has no labels, so test-set accuracy cannot be calculated.

## 1. Upload and extract the dataset

Upload a ZIP containing the `Butterfly/` dataset folder. If the data is already in Google Drive, skip this cell and update `DATA_DIR` in the configuration cell.

In [1]:
# Upload the dataset ZIP from your computer and extract it into /content.
from google.colab import files
import os
import zipfile

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)

if zip_name:
    # Extract the archive while preserving its folder structure.
    with zipfile.ZipFile(zip_name, "r") as zip_file:
        zip_file.extractall("/content")

    print(f"Extracted: {zip_name}")
    print("Items in /content:", os.listdir("/content"))
else:
    print("No ZIP uploaded. If your dataset is in Google Drive, mount Drive and set DATA_DIR accordingly.")

KeyboardInterrupt: 

## 2. Verify the dataset layout

The dataset directory should contain `train/`, `test/`, `Training_set.csv`, and `Testing_set.csv`. If the ZIP created a nested `Butterfly/Butterfly/` folder, update `DATA_DIR` below.


In [2]:
import zipfile
import os
from google.colab import drive
drive.mount('/content/drive')

# Update this path to match your Google Drive.
ZIP_PATH = "/content/drive/MyDrive/Butterfly.zip"

# Extract the dataset into Colab's temporary storage.
EXTRACT_DIR = "/content/butterfly_dataset"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    zip_file.extractall(EXTRACT_DIR)

print("Dataset extracted to:", EXTRACT_DIR)
print("Extracted items:", os.listdir(EXTRACT_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset extracted to: /content/butterfly_dataset
Extracted items: ['__MACOSX', 'Butterfly']


In [3]:
import os

# Change this path if your dataset was extracted somewhere else.
DATA_DIR = EXTRACT_DIR+"/Butterfly"

required_items = ["train", "test", "Training_set.csv", "Testing_set.csv"]
print("Dataset path:", DATA_DIR)

# Report each expected item so path issues are easy to spot.
for item in required_items:
    item_path = os.path.join(DATA_DIR, item)
    status = "OK" if os.path.exists(item_path) else "MISSING"
    print(f"{status:7} {item_path}")

if not all(os.path.exists(os.path.join(DATA_DIR, item)) for item in required_items):
    print("\nFix DATA_DIR or the ZIP folder layout before running training.")

Dataset path: /content/butterfly_dataset/Butterfly
OK      /content/butterfly_dataset/Butterfly/train
OK      /content/butterfly_dataset/Butterfly/test
OK      /content/butterfly_dataset/Butterfly/Training_set.csv
OK      /content/butterfly_dataset/Butterfly/Testing_set.csv


## 3. Install dependencies

In [4]:
!pip -q install torch torchvision pandas matplotlib pillow

## 4. Import libraries

These libraries support image loading, data preparation, model training, plotting, reproducibility, and saving results.

In [5]:
import argparse
import copy
import json
import os
import random
import time
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms

## 5. Reproducibility and CSV column detection

These helpers make experiments repeatable and locate the filename/label columns even when their capitalization varies.

In [6]:
def set_seed(seed=42):
    'Set Python and PyTorch random seeds for more reproducible runs.'

    random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)

In [7]:
def find_column(df, candidates, description):
    'Find a DataFrame column matching one of the candidate names, ignoring case and whitespace.'

    normalized = {str(col).strip().lower(): col for col in df.columns}

    for candidate in candidates:

        if candidate.lower() in normalized:

            return normalized[candidate.lower()]

    raise ValueError(

        f"Cannot identify {description} column. Columns found: {list(df.columns)}"

    )

## 6. Dataset classes

The training dataset returns an image and its numeric label. The test dataset returns an image and its filename so predictions can be matched to the submission rows.

In [8]:
class ButterflyTrainDataset(Dataset):
    'Load labeled butterfly images from a directory and return transformed tensors with class indices.'

    def __init__(self, image_dir, dataframe, image_col, label_col, class_to_idx, transform=None):

        self.image_dir = image_dir

        self.data = dataframe.reset_index(drop=True)

        self.image_col = image_col

        self.label_col = label_col

        self.class_to_idx = class_to_idx

        self.transform = transform



    def __len__(self):

        return len(self.data)



    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        filename = str(row[self.image_col]).strip()

        image_path = os.path.join(self.image_dir, filename)

        if not os.path.isfile(image_path):

            raise FileNotFoundError(f"Training image not found: {image_path}")



        image = Image.open(image_path).convert("RGB")

        label_name = str(row[self.label_col]).strip()

        label = self.class_to_idx[label_name]

        if self.transform:

            image = self.transform(image)

        return image, label

In [9]:
class ButterflyTestDataset(Dataset):
    'Load unlabeled test images and return each transformed image with its filename.'

    def __init__(self, image_dir, dataframe, image_col, transform=None):

        self.image_dir = image_dir

        self.data = dataframe.reset_index(drop=True)

        self.image_col = image_col

        self.transform = transform



    def __len__(self):

        return len(self.data)



    def __getitem__(self, idx):

        filename = str(self.data.iloc[idx][self.image_col]).strip()

        image_path = os.path.join(self.image_dir, filename)

        if not os.path.isfile(image_path):

            raise FileNotFoundError(f"Test image not found: {image_path}")



        image = Image.open(image_path).convert("RGB")

        if self.transform:

            image = self.transform(image)

        return image, filename

## 7. Image preprocessing and augmentation

Training images receive random augmentation; validation and test images use deterministic resizing and cropping. All images are normalized using ImageNet statistics for pretrained weights.

In [10]:
def make_transforms(image_size=224):
    'Create training augmentations and deterministic evaluation transforms.'

    mean = [0.485, 0.456, 0.406]

    std = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([

        transforms.RandomResizedCrop(image_size),

        transforms.RandomHorizontalFlip(),

        transforms.ToTensor(),

        transforms.Normalize(mean, std),

    ])

    eval_tf = transforms.Compose([

        transforms.Resize(256),

        transforms.CenterCrop(image_size),

        transforms.ToTensor(),

        transforms.Normalize(mean, std),

    ])

    return train_tf, eval_tf

## 8. Build data loaders

Reads the CSV files, creates a stratified training/validation split, maps class names to indices, and wraps datasets in PyTorch data loaders.

In [11]:
def build_dataloaders(data_dir, batch_size, num_workers, image_size, seed):
    'Read dataset metadata and return train, validation, and test data loaders plus class metadata.'

    train_dir = os.path.join(data_dir, "train")

    test_dir = os.path.join(data_dir, "test")

    train_csv_path = os.path.join(data_dir, "Training_set.csv")

    test_csv_path = os.path.join(data_dir, "Testing_set.csv")



    for path in (train_dir, test_dir, train_csv_path, test_csv_path):

        if not os.path.exists(path):

            raise FileNotFoundError(f"Required path not found: {path}")



    train_df = pd.read_csv(train_csv_path)

    test_df = pd.read_csv(test_csv_path)



    train_image_col = find_column(

        train_df, ["file_name", "filename", "image_name", "image name", "image"],

        "training filename"

    )

    label_col = find_column(

        train_df, ["label", "class", "class_name", "category"], "label"

    )

    test_image_col = find_column(

        test_df, ["image name", "image_name", "file_name", "filename", "image", "file"],

        "test filename"

    )



    train_df[train_image_col] = train_df[train_image_col].astype(str).str.strip()

    train_df[label_col] = train_df[label_col].astype(str).str.strip()

    test_df[test_image_col] = test_df[test_image_col].astype(str).str.strip()



    class_names = sorted(train_df[label_col].unique().tolist())

    class_to_idx = {name: i for i, name in enumerate(class_names)}

    if len(train_df) < 2:

        raise ValueError("At least two labeled training images are required.")



    train_tf, eval_tf = make_transforms(image_size)

    indices = list(range(len(train_df)))

    random.Random(seed).shuffle(indices)

    val_size = max(1, int(0.2 * len(indices)))

    val_size = min(val_size, len(indices) - 1)

    val_indices, train_indices = indices[:val_size], indices[val_size:]



    train_aug = ButterflyTrainDataset(

        train_dir, train_df, train_image_col, label_col, class_to_idx, train_tf

    )

    train_eval = ButterflyTrainDataset(

        train_dir, train_df, train_image_col, label_col, class_to_idx, eval_tf

    )

    test_dataset = ButterflyTestDataset(

        test_dir, test_df, test_image_col, eval_tf

    )



    train_subset = Subset(train_aug, train_indices)

    val_subset = Subset(train_eval, val_indices)

    loader_args = {

        "batch_size": batch_size,

        "num_workers": num_workers,

        "pin_memory": torch.cuda.is_available(),

    }



    train_loader = DataLoader(train_subset, shuffle=True, **loader_args)

    val_loader = DataLoader(val_subset, shuffle=False, **loader_args)

    test_loader = DataLoader(test_dataset, shuffle=False, **loader_args)



    print("Training CSV columns:", list(train_df.columns))

    print("Test CSV columns:", list(test_df.columns))

    print("Classes:", class_names)

    print(f"Training: {len(train_subset)} | Validation: {len(val_subset)} | Test: {len(test_dataset)}")

    return train_loader, val_loader, test_loader, class_names, test_image_col

## 9. Build the model

Loads a pretrained architecture and replaces its final classification layer with one sized for the butterfly classes.

In [12]:
def build_model(model_name, num_classes, pretrained=True):
    'Create a pretrained image-classification model with a task-specific output layer.'

    weights = "DEFAULT" if pretrained else None

    if model_name == "resnet18":

        model = models.resnet18(weights=weights)

        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "resnet34":

        model = models.resnet34(weights=weights)

        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "vgg19":

        model = models.vgg19(weights=weights)

        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)

    else:

        raise ValueError(f"Unknown model: {model_name}")

    return model

## 10. Training and validation loops

`run_epoch` performs one pass over a data loader. `train_model` repeats training and validation for multiple epochs and saves the best checkpoint.

In [13]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    'Run one training or evaluation epoch and return average loss and accuracy.'

    training = optimizer is not None

    model.train(training)

    total_loss, total_correct, total_count = 0.0, 0, 0



    for images, labels in loader:

        images = images.to(device, non_blocking=True)

        labels = labels.to(device, non_blocking=True)

        if training:

            optimizer.zero_grad(set_to_none=True)



        with torch.set_grad_enabled(training):

            outputs = model(images)

            loss = criterion(outputs, labels)

            if training:

                loss.backward()

                optimizer.step()



        n = labels.size(0)

        total_loss += loss.item() * n

        total_correct += (outputs.argmax(dim=1) == labels).sum().item()

        total_count += n



    return total_loss / total_count, total_correct / total_count

In [14]:
def train_model(model, train_loader, val_loader, device, epochs, lr, checkpoint_path):
    'Train for the requested epochs, retain the best validation checkpoint, and return training history.'

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer, mode="max", factor=0.5, patience=2

    )

    best_val_acc = -1.0

    best_state = copy.deepcopy(model.state_dict())

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}



    for epoch in range(1, epochs + 1):

        start = time.time()

        train_loss, train_acc = run_epoch(

            model, train_loader, criterion, device, optimizer

        )

        val_loss, val_acc = run_epoch(model, val_loader, criterion, device)

        scheduler.step(val_acc)



        history["train_loss"].append(train_loss)

        history["train_acc"].append(train_acc)

        history["val_loss"].append(val_loss)

        history["val_acc"].append(val_acc)



        print(

            f"Epoch {epoch}/{epochs} | train loss={train_loss:.4f}, acc={train_acc:.4f} | "

            f"val loss={val_loss:.4f}, acc={val_acc:.4f} | {time.time()-start:.1f}s"

        )

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            best_state = copy.deepcopy(model.state_dict())

            torch.save(best_state, checkpoint_path)



    model.load_state_dict(best_state)

    return model, history, best_val_acc

## 11. Test-set prediction

Runs inference on the unlabeled test set and writes predicted class names alongside image filenames.

In [15]:
def predict_test(model, test_loader, device, class_names, output_csv, filename_col):
    'Predict labels for test images and save filename/prediction pairs to a CSV file.'

    model.eval()

    output_rows = []

    for images, filenames in test_loader:

        images = images.to(device, non_blocking=True)

        predicted_ids = model(images).argmax(dim=1).cpu().tolist()

        for filename, class_id in zip(filenames, predicted_ids):

            output_rows.append({

                filename_col: filename,

                "predicted_label": class_names[class_id],

                "predicted_class_id": class_id,

            })

    pd.DataFrame(output_rows).to_csv(output_csv, index=False)

    print("Saved predictions:", output_csv)

## 12. Plot training history

Saves a plot of training/validation loss and validation accuracy for each model.

In [16]:
def save_history_plot(history, model_name, output_dir):
    'Plot and save loss and validation-accuracy curves from the training history.'

    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], label="Train")

    axes[0].plot(epochs, history["val_loss"], label="Validation")

    axes[0].set_title("Loss")

    axes[0].set_xlabel("Epoch")

    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="Train")

    axes[1].plot(epochs, history["val_acc"], label="Validation")

    axes[1].set_title("Accuracy")

    axes[1].set_xlabel("Epoch")

    axes[1].legend()

    fig.suptitle(model_name)

    fig.tight_layout()

    fig.savefig(os.path.join(output_dir, f"{model_name}_training_curves.png"), dpi=150)

    plt.close(fig)

## 13. Configure the experiment

Edit these settings before training. `PRETRAINED = True` starts from ImageNet weights; this is usually a strong baseline for transfer learning.

In [17]:
# Experiment settings — adjust these values to control training.
OUTPUT_DIR = "/content/drive/MyDrive/Butterfly_outputs"
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224
NUM_WORKERS = 2
SEED = 42
PRETRAINED = True

# Create an output folder and select GPU when available.
import os
import torch

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cpu":
    print("Tip: In Colab, select Runtime → Change runtime type → T4 GPU for faster training.")

Using device: cuda


## 14. Prepare the data

This cell loads metadata, creates the train/validation split, and prints the discovered classes. Resolve any dataset-path or CSV-column errors before training.

In [18]:
train_loader, val_loader, test_loader, class_names, test_image_col = build_dataloaders(
    DATA_DIR,
    BATCH_SIZE,
    NUM_WORKERS,
    IMAGE_SIZE,
    SEED,
)

print(f"Number of classes: {len(class_names)}")
print("Classes:", class_names)
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Training CSV columns: ['filename', 'label']
Test CSV columns: ['filename']
Classes: ['ADONIS', 'AFRICAN GIANT SWALLOWTAIL', 'AMERICAN SNOOT', 'AN 88', 'APPOLLO', 'ATALA', 'BANDED ORANGE HELICONIAN', 'BANDED PEACOCK', 'BECKERS WHITE', 'BLACK HAIRSTREAK', 'BLUE MORPHO', 'BLUE SPOTTED CROW', 'BROWN SIPROETA', 'CABBAGE WHITE', 'CAIRNS BIRDWING', 'CHECQUERED SKIPPER', 'CHESTNUT', 'CLEOPATRA', 'CLODIUS PARNASSIAN', 'CLOUDED SULPHUR', 'COMMON BANDED AWL', 'COMMON WOOD-NYMPH', 'COPPER TAIL', 'CRECENT', 'CRIMSON PATCH', 'DANAID EGGFLY', 'EASTERN COMA', 'EASTERN DAPPLE WHITE', 'EASTERN PINE ELFIN', 'ELBOWED PIERROT', 'GOLD BANDED', 'GREAT EGGFLY', 'GREAT JAY', 'GREEN CELLED CATTLEHEART', 'GREY HAIRSTREAK', 'INDRA SWALLOW', 'IPHICLUS SISTER', 'JULIA', 'LARGE MARBLE', 'MALACHITE', 'MANGROVE SKIPPER', 'MESTRA', 'METALMARK', 'MILBERTS TORTOISESHELL', 'MONARCH', 'MOURNING CLOAK', 'ORANGE OAKLEAF', 'ORANGE TIP', 'ORCHARD SWALLOW', 'PAINTED LADY', 'PAPER KITE', 'PEACOCK', 'PINE WHITE', 'PIPEVINE SWALLO

## 15. Train and evaluate the three architectures

Each model is trained independently using the same data split and settings. For each architecture, the notebook saves the best checkpoint, plots the learning curves, and creates a test-prediction CSV.

In [19]:
# Store summary information for each architecture as training completes.
results = {}

model_names = ("resnet18", "resnet34", "vgg19")

for model_name in model_names:
    print(f"\n{'=' * 60}\nTraining {model_name}\n{'=' * 60}")

    # Create the architecture and move it to the selected device.
    model = build_model(
        model_name,
        num_classes=len(class_names),
        pretrained=PRETRAINED,
    ).to(device)

    # Train the model and save its best validation checkpoint.
    checkpoint_path = os.path.join(OUTPUT_DIR, f"{model_name}_best.pth")
    model, history, best_val_acc = train_model(
        model,
        train_loader,
        val_loader,
        device,
        EPOCHS,
        LEARNING_RATE,
        checkpoint_path,
    )

    # Save the learning curves for later comparison.
    save_history_plot(history, model_name, OUTPUT_DIR)

    # Generate predictions for the unlabeled test images.
    prediction_path = os.path.join(
        OUTPUT_DIR, f"{model_name}_test_predictions.csv"
    )
    predict_test(
        model,
        test_loader,
        device,
        class_names,
        prediction_path,
        test_image_col,
    )

    # Test accuracy is unavailable because the test CSV has no ground-truth labels.
    results[model_name] = {
        "best_validation_accuracy": best_val_acc,
        "checkpoint": checkpoint_path,
        "test_predictions": prediction_path,
        "test_accuracy": None,
        "note": "Test CSV has no labels; test accuracy cannot be computed.",
    }


Training resnet18
Epoch 1/10 | train loss=2.2794, acc=0.5225 | val loss=0.6689, acc=0.8453 | 25.6s
Epoch 2/10 | train loss=0.9643, acc=0.7856 | val loss=0.3922, acc=0.9107 | 25.0s
Epoch 3/10 | train loss=0.6826, acc=0.8352 | val loss=0.3187, acc=0.9222 | 26.5s
Epoch 4/10 | train loss=0.5723, acc=0.8646 | val loss=0.3116, acc=0.9261 | 27.0s
Epoch 5/10 | train loss=0.5044, acc=0.8763 | val loss=0.2803, acc=0.9299 | 27.0s
Epoch 6/10 | train loss=0.4262, acc=0.8948 | val loss=0.3108, acc=0.9192 | 27.0s
Epoch 7/10 | train loss=0.4106, acc=0.8956 | val loss=0.2780, acc=0.9353 | 26.1s
Epoch 8/10 | train loss=0.3492, acc=0.9092 | val loss=0.2641, acc=0.9392 | 27.1s
Epoch 9/10 | train loss=0.3452, acc=0.9081 | val loss=0.2855, acc=0.9323 | 26.7s
Epoch 10/10 | train loss=0.3366, acc=0.9115 | val loss=0.2591, acc=0.9361 | 26.9s
Saved predictions: /content/drive/MyDrive/Butterfly_outputs/resnet18_test_predictions.csv

Training resnet34
Epoch 1/10 | train loss=1.9741, acc=0.5648 | val loss=0.5064,

100%|██████████| 548M/548M [00:05<00:00, 98.7MB/s]


Epoch 1/10 | train loss=3.9176, acc=0.0604 | val loss=2.9044, acc=0.1894 | 129.5s
Epoch 2/10 | train loss=2.6991, acc=0.2608 | val loss=1.7197, acc=0.5273 | 128.6s
Epoch 3/10 | train loss=1.8766, acc=0.4548 | val loss=1.1867, acc=0.6313 | 128.2s
Epoch 4/10 | train loss=1.4872, acc=0.5713 | val loss=0.9478, acc=0.7159 | 128.3s
Epoch 5/10 | train loss=1.2639, acc=0.6304 | val loss=0.7757, acc=0.7629 | 127.7s
Epoch 6/10 | train loss=1.1037, acc=0.6740 | val loss=0.7699, acc=0.7798 | 128.1s
Epoch 7/10 | train loss=1.0111, acc=0.7044 | val loss=0.6773, acc=0.8060 | 128.1s
Epoch 8/10 | train loss=0.9407, acc=0.7244 | val loss=0.6069, acc=0.8507 | 128.0s
Epoch 9/10 | train loss=0.8560, acc=0.7487 | val loss=0.6283, acc=0.8376 | 128.0s
Epoch 10/10 | train loss=0.8198, acc=0.7592 | val loss=0.5983, acc=0.8337 | 127.7s
Saved predictions: /content/drive/MyDrive/Butterfly_outputs/vgg19_test_predictions.csv


## 16. Save and review results

The JSON summary records each model's best validation accuracy and the paths to its checkpoint and prediction file.

In [20]:
# Save the experiment summary in a machine-readable format.
results_path = os.path.join(OUTPUT_DIR, "results.json")
with open(results_path, "w", encoding="utf-8") as results_file:
    json.dump(results, results_file, indent=2)

print("Best validation accuracies:")
for model_name, result in results.items():
    print(f"{model_name}: {result['best_validation_accuracy'] * 100:.2f}%")

print("\nResults saved to:", results_path)
print("Test labels are not present; predictions were generated without test accuracy.")

Best validation accuracies:
resnet18: 93.92%
resnet34: 93.38%
vgg19: 85.07%

Results saved to: /content/drive/MyDrive/Butterfly_outputs/results.json
Test labels are not present; predictions were generated without test accuracy.


## 17. Output files

After all cells finish, find the files in `/content/outputs/`:

- `resnet18_best.pth`, `resnet34_best.pth`, `vgg19_best.pth` — best model checkpoints.
- `*_test_predictions.csv` — predicted class for each test filename.
- `*_history.png` — training/validation curves for each architecture.
- `results.json` — summary of validation scores and output paths.

**Tip:** Download or copy the output folder before the Colab runtime disconnects, because temporary runtime files may be deleted.